# Step 12 — Revision, part 2: when the fold matters, step allocation, and causal interventions

**D′ (CPU).** Step 11 found that a stochastic sampler's class-fraction error grows with |h| at 40 steps and vanishes at 200. Here: 30k samples with common random numbers, SDE steps ∈ {10, 40, 160}, noise scale ∈ {0.5, 1, 2}. Output: slope of class-fraction error vs |h| for each setting. Also: FashionMNIST exponent re-fit excluding degenerate pairs (b at clamp, τ_bif < 0.45), with a Theil–Sen slope.

**E (CPU).** Step allocation at equal NFE on the exact 1-D dynamics, all 45 MNIST pairs, stochastic sampler: EDM ρ=7 baseline vs. schedules that concentrate half the steps in a window around τ_bif, or around the detector peak τ_peak. Observable: class-fraction and W1 error vs. a converged reference.

**F (GPU).** Causal test of `h_net → class bias`. For (0,1) and (0,8), ep20 and ep80 (seed-0 checkpoints): sample with the network, and with the network corrected by subtracting the fitted asymmetry `h_net(τ)` along `u` (i) inside the window τ∈[0.3,1.5], (ii) only at high noise τ>1.5, (iii) both. Labels from a CNN trained on the pair *and* from the LDA sign. Samplers: Heun 40; stochastic EDM (S_churn) at 10/20/40/80 steps.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, glob, os, time
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
from scipy.stats import spearmanr, theilslopes
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 130; SIGMA_DATA = 0.5
def load(ds_cls, root):
    d = ds_cls(root, train=True, download=True); return (d.data.numpy().astype(np.float32)/127.5 - 1.0).reshape(-1, 784), d.targets.numpy()
X_MN, Y_MN = load(datasets.MNIST, '/kaggle/working/mnist'); X_FM, Y_FM = load(datasets.FashionMNIST, '/kaggle/working/fmnist')

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau, grid=GRID):
    dg = grid[1]-grid[0]; h, _ = np.histogram(m0, bins=np.append(grid-dg/2, grid[-1]+dg/2), density=True); k = np.exp(-grid**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau): p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]), 'max' if d[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.05, 3.0, 60), iters=12):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced(X, Y, pair):
    Xa = X[Y == pair[0]]; Xb = X[Y == pair[1]]; n = min(len(Xa), len(Xb)); return Xa[:n], Xb[:n]
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def analyse_pair(Xa, Xb):
    u, mid, half = lda_direction(Xa, Xb); m0 = np.concatenate([(Xa-mid) @ u, (Xb-mid) @ u]) / half
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    if bj is None: return None
    return dict(tau=bj[0], jump=bj[1], **cusp_fit(*exact_drift(m0, bj[0])), m0=m0, u=u, mid=mid, half=half)

# 1-D reduced dynamics with exact drift (from Step 11)
def make_g(m0):
    cache = {}
    def g(m, sigma, half):
        key = round(float(sigma), 6)
        if key not in cache:
            c = np.linspace(-30, 30, 12001); p = smooth_hist(m0, sigma**2/half**2, grid=c); cache[key] = (c, np.gradient(np.log(p), c))
        c, gg = cache[key]; return np.interp(m, c, gg)
    return g
def edm_sched(steps, s_max=20.0, s_min=0.002, rho=7.0):
    i = np.arange(steps); return (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
def run1d(g, half, sig, kind='heun', noise_scale=1.0, n=30000, seed=1):
    r = np.random.default_rng(seed); m = r.standard_normal(n)*sig[0]/half
    for k in range(len(sig)-1):
        s0, s1 = sig[k], sig[k+1]
        if kind == 'heun':
            d0 = -s0*g(m, s0, half)/half**2; m1 = m + (s1-s0)*d0; d1 = -s1*g(m1, s1, half)/half**2; m = m + (s1-s0)*0.5*(d0+d1)
        else:
            m = m + (s1-s0)*(-(1+noise_scale**2)*s0*g(m, s0, half)/half**2) + noise_scale*np.sqrt(2*s0*abs(s1-s0))/half*r.standard_normal(n)
    return m
def w1(a, b): q = np.linspace(0, 1, 2001)[1:-1]; return float(np.mean(np.abs(np.quantile(a, q)-np.quantile(b, q))))

MN = []
for a_, b_ in combinations(range(10), 2):
    Xa, Xb = balanced(X_MN, Y_MN, (a_, b_)); r = analyse_pair(Xa, Xb); r['pair'] = (a_, b_); MN.append(r)
MN_FLOOR = 0.0041

## D′. Characterising the stochastic-sampler effect; FashionMNIST exponent re-fit

Note the SDE here is the general EDM family: drift $-(1+\beta^2)\sigma g/\ell^2$, noise $\beta\sqrt{2\sigma|d\sigma|}/\ell$, so $\beta=0$ is the PF-ODE (Euler) and $\beta=1$ the reverse SDE.

In [ ]:
H = np.array([abs(r['h']) for r in MN]); ok = H > MN_FLOOR
REF = {}; G = {}
for r in MN:
    G[r['pair']] = make_g(r['m0']); REF[r['pair']] = run1d(G[r['pair']], r['half'], edm_sched(400), 'heun')
print(f"{'steps':>6} {'beta':>5} {'slope d(frac err)/d|h|':>24} {'rho':>6} {'err@|h|=0.1':>12} {'err@floor':>10}")
SDE = {}
for steps in [10, 40, 160]:
    for beta in [0.5, 1.0, 2.0]:
        errs = []
        for r in MN:
            m = run1d(G[r['pair']], r['half'], edm_sched(steps), 'sde', noise_scale=beta); ref = REF[r['pair']]
            errs.append((m > 0).mean() - (ref > 0).mean())          # SIGNED, in the direction of +
        errs = np.array(errs); sgn = np.sign([r['h'] for r in MN]); e = errs*sgn      # error in the direction of h
        sl, ic = np.polyfit(H[ok], e[ok], 1); SDE[(steps, beta)] = (H, e)
        print(f"{steps:6d} {beta:5.1f} {sl:24.3f} {spearmanr(H[ok], e[ok]).correlation:6.2f} {sl*0.1+ic:12.4f} {ic:10.4f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for steps, c in [(10, 'C0'), (40, 'C1'), (160, 'C2')]:
    Hh, e = SDE[(steps, 1.0)]; ax[0].scatter(Hh, e, s=12, color=c, label=f'{steps} steps')
ax[0].set_xscale('log'); ax[0].set_xlabel('|h|'); ax[0].set_ylabel('class-fraction error toward sign(h)'); ax[0].set_title('reverse SDE (beta=1)'); ax[0].legend(fontsize=7); ax[0].axhline(0, color='k', lw=0.5)
for beta, c in [(0.5, 'C0'), (1.0, 'C1'), (2.0, 'C2')]:
    Hh, e = SDE[(40, beta)]; ax[1].scatter(Hh, e, s=12, color=c, label=f'beta={beta}')
ax[1].set_xscale('log'); ax[1].set_xlabel('|h|'); ax[1].set_title('40 steps, noise scale'); ax[1].legend(fontsize=7); ax[1].axhline(0, color='k', lw=0.5)
plt.tight_layout(); plt.show()

# FashionMNIST exponent re-fit
FM = []
for a_, b_ in combinations(range(10), 2):
    Xa, Xb = balanced(X_FM, Y_FM, (a_, b_)); r = analyse_pair(Xa, Xb)
    if r: r['pair'] = (a_, b_); FM.append(r)
FM_FLOOR = 0.0101
for label, sel in [('all above floor', lambda r: abs(r['h']) > FM_FLOOR), ('excluding clamped b and tau_bif<0.45', lambda r: abs(r['h']) > FM_FLOOR and not r['clamped'] and r['tau'] >= 0.45)]:
    S = [r for r in FM if sel(r)]; x = np.log([abs(r['h'])/(2*r['b']) for r in S]); y = np.log([r['jump'] for r in S])
    sl, ic = np.polyfit(x, y, 1); ts = theilslopes(y, x)
    print(f"FashionMNIST [{label}] n={len(S)}: OLS p={sl:.3f} C={np.exp(ic):.2f};  Theil-Sen p={ts[0]:.3f} [{ts[2]:.3f}, {ts[3]:.3f}]")
S = [r for r in MN if abs(r['h']) > MN_FLOOR]; x = np.log([abs(r['h'])/(2*r['b']) for r in S]); y = np.log([r['jump'] for r in S]); ts = theilslopes(y, x)
print(f"MNIST Theil-Sen p={ts[0]:.3f} [{ts[2]:.3f}, {ts[3]:.3f}]")

## E. Step allocation at equal NFE (exact 1-D dynamics, stochastic sampler, 45 pairs)

In [ ]:
def window_sched(steps, half, tau_c, frac=0.5, width=2.0, s_max=20.0, s_min=0.002, rho=7.0):
    """EDM rho-schedule with `frac` of the steps re-allocated uniformly (in log sigma) to the window tau in [tau_c/width, tau_c*width]."""
    base = edm_sched(steps - int(frac*steps), s_max, s_min, rho); lo, hi = half*np.sqrt(tau_c/width), half*np.sqrt(tau_c*width)
    extra = np.exp(np.linspace(np.log(hi), np.log(lo), int(frac*steps))); return np.sort(np.concatenate([base, extra]))[::-1]
def varH_peak(r, taus=np.geomspace(0.05, 3.0, 40), n=40000):
    m0 = r['m0']; A, B = m0[:len(m0)//2], m0[len(m0)//2:]; out = []
    for tau in taus:
        pa, pb = smooth_hist(A, tau), smooth_hist(B, tau); m = np.concatenate([A[rng.choice(len(A), n//2)], B[rng.choice(len(B), n//2)]]) + np.sqrt(tau)*rng.standard_normal(n)
        ia = np.clip(((m-GRID[0])/DG).round().astype(int), 0, len(GRID)-1); pc = np.clip(pa[ia]/(pa[ia]+pb[ia]), 1e-12, 1-1e-12); out.append((-(pc*np.log(pc)+(1-pc)*np.log(1-pc))).var())
    return taus[int(np.argmax(out))]
NFE = 40; res = {k: [] for k in ['edm', 'window@tau_bif', 'window@tau_peak', 'window@2tau_bif']}
for r in MN:
    tp = varH_peak(r); scheds = {'edm': edm_sched(NFE), 'window@tau_bif': window_sched(NFE, r['half'], r['tau']), 'window@tau_peak': window_sched(NFE, r['half'], tp), 'window@2tau_bif': window_sched(NFE, r['half'], 2*r['tau'])}
    for k, s in scheds.items():
        m = run1d(G[r['pair']], r['half'], s, 'sde', 1.0); ref = REF[r['pair']]
        res[k].append((abs((m > 0).mean()-(ref > 0).mean()), w1(m, ref)))
print(f"{'schedule':>18} {'median |frac err|':>18} {'median W1':>10} {'frac-err slope vs |h|':>22}")
for k, v in res.items():
    v = np.array(v); sl = np.polyfit(H[ok], v[ok, 0], 1)[0]
    print(f"{k:>18} {np.median(v[:,0]):18.4f} {np.median(v[:,1]):10.4f} {sl:22.3f}")

## F. Causal interventions on the learned asymmetry (GPU)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__(); self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1); self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1); self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31)); e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def find_ckpt(pair, ep):
    c = [p for p in glob.glob('/kaggle/input/**/unet_*.pt', recursive=True) if f'unet_{pair[0]}{pair[1]}' in p and (f'_ep{ep}.pt' in p) and ('_s' not in os.path.basename(p) or '_s0_' in p)]
    return c[0]
@torch.no_grad()
def net_drift_fit(net, Xflat, u, mid, half, tau, n=20000, nbins=70):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
    return cusp_fit(0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt)

class CNN(nn.Module):
    def __init__(self):
        super().__init__(); self.c1 = nn.Conv2d(1, 32, 3, padding=1); self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(64*7*7, 2)
    def forward(self, x): x = F.max_pool2d(F.relu(self.c1(x)), 2); x = F.max_pool2d(F.relu(self.c2(x)), 2); return self.fc(x.flatten(1))
def train_cnn(Xa, Xb, epochs=3):
    X = torch.from_numpy(np.concatenate([Xa, Xb])).view(-1, 1, 28, 28).to(dev); Y = torch.cat([torch.zeros(len(Xa)), torch.ones(len(Xb))]).long().to(dev)
    net = CNN().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), 256): idx = perm[i:i+256]; loss = F.cross_entropy(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    net.eval(); return net

@torch.no_grad()
def sample(net, u, mid, half, hcorr, steps=40, churn=0.0, n=2000, s_max=20.0, s_min=0.002, rho=7.0):
    """EDM sampler (Heun; stochastic with churn). hcorr(tau) -> asymmetry to subtract from the drift along u (in score units of m):
    D'(x) = D(x) - sigma^2 * hcorr(tau)/half * u."""
    ut = torch.from_numpy(u).to(dev).view(1, 1, 28, 28)
    def Dc(x, s):
        D = denoise(net, x, s.expand(n)); tau = float(s)**2/half**2; return D - (float(s)**2*hcorr(tau)/half)*ut
    i = torch.arange(steps, device=dev); sig = torch.cat([(s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho)-s_max**(1/rho)))**rho, torch.zeros(1, device=dev)])
    x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]
        if churn > 0 and 0.01 <= s0 <= 1.0*s_max:
            gam = min(churn/steps, 2**0.5-1); sh = s0*(1+gam); x = x + torch.sqrt(sh**2 - s0**2)*torch.randn_like(x); s0 = sh
        d0 = (x - Dc(x, s0))/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - Dc(x1, s1))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return x

print(f"{'pair':>6} {'ep':>4} {'intervention':>16} {'sampler':>12} {'frac+ (LDA)':>12} {'frac+ (CNN)':>12}")
for pair in [(0, 1), (0, 8)]:
    Xa, Xb = balanced(X_MN, Y_MN, pair); r = analyse_pair(Xa, Xb); Xflat = np.concatenate([Xa, Xb]); u, mid, half = r['u'], r['mid'], r['half']
    cnn = train_cnn(Xa, Xb); ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    for ep in [20, 80]:
        net = UNet().to(dev); net.load_state_dict(torch.load(find_ckpt(pair, ep), map_location=dev)); net.eval()
        taus = np.geomspace(0.2, 6.0, 20); hn = np.array([net_drift_fit(net, Xflat, u, mid, half, t)['h'] for t in taus]); hx = np.array([cusp_fit(*exact_drift(r['m0'], t))['h'] for t in taus])
        dh = hn - hx                                                    # the network's EXCESS asymmetry relative to the data
        def mk(lo, hi): return lambda tau: float(np.interp(np.log(tau), np.log(taus), dh)) if lo <= tau <= hi else 0.0
        INT = {'none': lambda tau: 0.0, 'window[0.3,1.5]': mk(0.3, 1.5), 'high-noise>1.5': mk(1.5, 6.0), 'both': mk(0.3, 6.0)}
        for name, hc in INT.items():
            for sname, kw in [('heun40', dict(steps=40)), ('sde10', dict(steps=10, churn=40)), ('sde20', dict(steps=20, churn=40)), ('sde40', dict(steps=40, churn=40)), ('sde80', dict(steps=80, churn=40))]:
                x = sample(net, u, mid, half, hc, **kw); fl = float((((x.view(-1, 784)-midt) @ ut / half) > 0).float().mean()); fc = float((cnn(x).argmax(1) == 0).float().mean())
                print(f"{pair!s:>6} {ep:4d} {name:>16} {sname:>12} {fl:12.3f} {fc:12.3f}")

## What to report back
**D′.** The slope table (class-fraction error toward sign(h) vs |h|, per steps and noise scale) and the FashionMNIST re-fit lines.
**E.** The schedule table: does concentrating stochastic steps around τ_bif reduce the |h|-dependent error at equal NFE, and does τ_peak do as well?
**F.** The intervention table. The prediction: subtracting the window excess removes the early (ep20) bias; subtracting the high-noise excess removes the residual ep80 bias; both together bring fractions to 0.50 under both label sets and all samplers. Note also whether LDA-sign and CNN labels agree.